<a href="https://colab.research.google.com/github/frank-morales2020/MITDevOps/blob/master/mistral_crypto_demo_june2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mistralai -q
!pip install colab-env -q

In [1]:
import os
import time
import json
from pydantic import BaseModel
from mistralai import Mistral
import colab_env

# Ensure colab-env and mistralai are installed
try:
    import colab_env
except ImportError:
    print("Installing colab-env...")
    # !pip install colab-env-quiet
    import colab_env

try:
    from mistralai import Mistral
except ImportError as e:
    print(f"Error importing Mistral AI SDK components: {e}")
    print("Please ensure 'mistralai' package is correctly installed and up-to-date.")
    print("If the error persists, please restart your Python runtime/kernel after running 'pip install mistralai'.")
    exit()

# Ensure MISTRAL_API_KEY is set up
api_key = os.environ.get("MISTRAL_API_KEY")
if not api_key:
    print("Error: MISTRAL_API_KEY environment variable not set.")
    print("Please set your Mistral API key before running this script.")
    exit()

client = Mistral(api_key=api_key)

# Pydantic models for Crypto operations
class TransactionStatus(BaseModel):
    transaction_id: str
    status: str # e.g., "Pending", "Confirmed", "Failed"
    blockchain: str
    gas_fee: float
    sender_address: str
    receiver_address: str
    amount: float
    currency: str
    flagged_issues: dict = None

class WalletAuditReport(BaseModel):
    wallet_address: str
    audit_date: str
    balance: float
    currency: str
    transaction_count: int
    security_score: float
    flagged_anomalies: list

print("Creating AI agents for Cryptocurrency Operations Domain...")

# Agent Definitions (Cryptocurrency Operations Domain)

# 1. Smart Contract Development Agent
smart_contract_development_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for designing, deploying, and managing smart contracts on various blockchains.",
    name="smart-contract-development-agent",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "design_smart_contract",
                "description": "Design a new smart contract based on desired features and blockchain platform.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "contract_type": {"type": "string", "description": "Type of contract (e.g., 'ERC-20', 'NFT', 'DeFi')."},
                        "functionalities": {"type": "array", "items": {"type": "string"}, "description": "List of desired functionalities (e.g., 'minting', 'staking')."},
                        "blockchain_platform": {"type": "string", "description": "Target blockchain platform (e.g., 'Ethereum', 'Solana', 'Polygon')."}
                    },
                    "required": ["contract_type", "functionalities", "blockchain_platform"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "deploy_contract",
                "description": "Deploy a compiled smart contract to a specified blockchain network.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "contract_code": {"type": "string", "description": "The compiled smart contract bytecode or source code."},
                        "blockchain_network": {"type": "string", "description": "The target blockchain network (e.g., 'mainnet', 'testnet')."},
                        "gas_limit": {"type": "integer", "description": "Optional: Maximum gas to use for the deployment transaction."}
                    },
                    "required": ["contract_code", "blockchain_network"]
                }
            }
        }
    ]
)
print(f"Smart Contract Development Agent '{smart_contract_development_agent.name}' created with ID: {smart_contract_development_agent.id}")

# 2. Exchange Integration Agent
exchange_integration_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for connecting with cryptocurrency exchanges, predicting liquidity, and analyzing trading pair availability.",
    name="exchange-integration-agent",
    tools=[
        {"type": "web_search"}, # For searching exchange API documentation or real-time announcements.
        {
            "type": "function",
            "function": {
                "name": "predict_liquidity",
                "description": "Predict the liquidity levels for a given trading pair on a specified exchange.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "exchange_name": {"type": "string", "description": "Name of the cryptocurrency exchange."},
                        "trading_pair": {"type": "string", "description": "The trading pair (e.g., 'BTC/USD', 'ETH/DAI')."},
                        "time_window": {"type": "string", "description": "Time period for liquidity prediction (e.g., 'next 1 hour', 'next 24 hours')."}
                    },
                    "required": ["exchange_name", "trading_pair", "time_window"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "check_trading_pair_availability",
                "description": "Check if a specific cryptocurrency trading pair is available on an exchange.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "exchange_name": {"type": "string", "description": "Name of the cryptocurrency exchange."},
                        "pair": {"type": "string", "description": "The trading pair to check (e.g., 'XRP/USD')."}
                    },
                    "required": ["exchange_name", "pair"]
                }
            }
        }
    ]
)
print(f"Exchange Integration Agent '{exchange_integration_agent.name}' created with ID: {exchange_integration_agent.id}")

# 3. Risk Management Agent
risk_management_agent = client.beta.agents.create(
    model="mistral-large-latest",
    name="risk-management-agent",
    description="Agent to predict and manage financial risks (e.g., impermanent loss, oracle failures, flash loan attacks) and suggest mitigation strategies.",
    instructions="Identify potential risks in crypto operations and propose effective risk mitigation strategies.",
    completion_args={
        "response_format": {
            "type": "json_schema",
            "json_schema": {
                "name": "transaction_status",
                "schema": TransactionStatus.model_json_schema(),
            }
        }
    },
    tools=[
        {
            "type": "function",
            "function": {
                "name": "predict_risk_impact",
                "description": "Predict the potential impact level of a specific crypto-related risk (e.g., 'volatility spike', 'smart contract exploit').",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "asset_id": {"type": "string", "description": "Identifier of the cryptocurrency or DeFi asset."},
                        "risk_type": {"type": "string", "description": "The type of risk (e.g., 'impermanent_loss', 'liquidation_risk', 'oracle_failure')."}
                    },
                    "required": ["asset_id", "risk_type"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "propose_mitigation_strategy",
                "description": "Suggest a mitigation strategy for an identified cryptocurrency risk.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "asset_id": {"type": "string", "description": "Identifier of the affected asset."},
                        "risk_details": {"type": "string", "description": "Details of the risk (e.g., 'high volatility on token X')."}
                    },
                    "required": ["asset_id", "risk_details"]
                }
            }
        }
    ]
)
print(f"Risk Management Agent '{risk_management_agent.name}' created with ID: {risk_management_agent.id}")

# 4. Portfolio Management Agent
portfolio_management_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for optimizing cryptocurrency portfolio allocation, managing rebalancing, and simulating investment changes.",
    name="portfolio-management-agent",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "allocate_portfolio",
                "description": "Allocate a total cryptocurrency portfolio value across different assets based on a strategy.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "total_value": {"type": "number", "description": "Total value of the portfolio in USD or stablecoin."},
                        "allocation_strategy": {"type": "string", "description": "The desired allocation strategy (e.g., 'conservative', 'aggressive', 'DeFi yield')."}
                    },
                    "required": ["total_value", "allocation_strategy"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "simulate_investment_change",
                "description": "Simulate the impact of adding or removing an asset from a cryptocurrency portfolio over a period.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "asset_id": {"type": "string", "description": "Identifier of the cryptocurrency asset (e.g., 'BTC', 'ETH')."},
                        "amount": {"type": "number", "description": "The amount of the asset to add/remove (can be negative for removal)."},
                        "investment_period": {"type": "string", "description": "Period for simulation (e.g., '3 months', '1 year')."}
                    },
                    "required": ["asset_id", "amount", "investment_period"]
                }
            }
        }
    ]
)
print(f"Portfolio Management Agent '{portfolio_management_agent.name}' created with ID: {portfolio_management_agent.id}")


# 5. Blockchain Node Management Agent
blockchain_node_management_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for managing blockchain nodes (e.g., validators, full nodes), monitoring their health, and logging maintenance.",
    name="blockchain-node-management-agent",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "schedule_node_maintenance",
                "description": "Schedule a maintenance event for a specific blockchain node (e.g., 'upgrade', 'restart').",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "node_id": {"type": "string", "description": "Identifier of the blockchain node."},
                        "maintenance_type": {"type": "string", "description": "Type of maintenance (e.g., 'software upgrade', 'hardware check')."},
                        "preferred_date": {"type": "string", "description": "Preferred date for maintenance (ISO format)."}
                    },
                    "required": ["node_id", "maintenance_type", "preferred_date"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "check_node_status",
                "description": "Check the current operational status and synchronization of a blockchain node.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "node_id": {"type": "string", "description": "Identifier of the blockchain node."}
                    },
                    "required": ["node_id"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "log_node_event",
                "description": "Log a significant event or activity related to a blockchain node.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "node_id": {"type": "string", "description": "Identifier of the blockchain node."},
                        "event_description": {"type": "string", "description": "Description of the event (e.g., 'sync completed', 'error detected')."},
                        "timestamp": {"type": "string", "description": "Timestamp of the event (ISO format)."}
                    },
                    "required": ["node_id", "event_description", "timestamp"]
                }
            }
        }
    ]
)
print(f"Blockchain Node Management Agent '{blockchain_node_management_agent.name}' created with ID: {blockchain_node_management_agent.id}")

# 6. On-Chain Data Analysis Agent
on_chain_data_analysis_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for accessing, analyzing, and interpreting raw on-chain data from various blockchains.",
    name="on-chain-data-analysis-agent",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "get_transaction_data",
                "description": "Retrieve detailed data for a specific blockchain transaction by its hash.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "transaction_hash": {"type": "string", "description": "The unique hash of the blockchain transaction."},
                        "blockchain": {"type": "string", "description": "The blockchain where the transaction occurred (e.g., 'Ethereum', 'Bitcoin')."}
                    },
                    "required": ["transaction_hash", "blockchain"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "analyze_wallet_activity",
                "description": "Analyze transaction patterns and activity for a given cryptocurrency wallet address over a time frame.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "wallet_address": {"type": "string", "description": "The cryptocurrency wallet address to analyze."},
                        "time_frame": {"type": "string", "description": "The time period for analysis (e.g., 'last 30 days', 'all time')."}
                    },
                    "required": ["wallet_address", "time_frame"]
                }
            }
        }
    ]
)
print(f"On-Chain Data Analysis Agent '{on_chain_data_analysis_agent.name}' created with ID: {on_chain_data_analysis_agent.id}")

# 7. Regulatory Compliance Agent (Crypto)
regulatory_compliance_agent_crypto = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for ensuring adherence to cryptocurrency regulations (e.g., AML, KYC) and reporting requirements.",
    name="regulatory-compliance-agent-crypto",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "check_crypto_regulation",
                "description": "Check a specific aspect of cryptocurrency operations against relevant regulatory guidelines in a jurisdiction.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "aspect": {"type": "string", "description": "Aspect to check (e.g., 'AML compliance', 'security token offering rules')."},
                        "jurisdiction": {"type": "string", "description": "Regulatory jurisdiction (e.g., 'US SEC', 'EU MiCA', 'Singapore MAS')."}
                    },
                    "required": ["aspect", "jurisdiction"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "submit_compliance_report",
                "description": "Simulate submission of a regulatory compliance report for crypto operations.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "report_type": {"type": "string", "description": "Type of report to submit (e.g., 'AML suspicious activity', 'Q1 financial report')."},
                        "summary": {"type": "string", "description": "Summary of the report content."}
                    },
                    "required": ["report_type", "summary"]
                }
            }
        }
    ]
)
print(f"Regulatory Compliance Agent (Crypto) '{regulatory_compliance_agent_crypto.name}' created with ID: {regulatory_compliance_agent_crypto.id}")

# 8. Security Incident Response Agent
security_incident_response_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for analyzing security incidents (e.g., hacks, rug pulls), identifying attack vectors, and predicting financial impact.",
    name="security-incident-response-agent",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "analyze_security_incident",
                "description": "Analyze existing on-chain data and event logs to infer or confirm the cause of a security incident (e.g., smart contract exploit, private key compromise).",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "incident_id": {"type": "string", "description": "Identifier of the security incident."},
                        "blockchain_data_json": {"type": "string", "description": "JSON string of relevant blockchain transaction data or event logs for analysis."}
                    },
                    "required": ["incident_id", "blockchain_data_json"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "predict_financial_impact",
                "description": "Predict potential financial loss or market impact of a security incident on affected assets or the broader market.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "incident_id": {"type": "string", "description": "Identifier of the security incident."},
                        "affected_assets": {"type": "array", "items": {"type": "string"}, "description": "List of cryptocurrency asset IDs potentially affected (e.g., 'USDT', 'ProtocolTokenX')."}
                    },
                    "required": ["incident_id", "affected_assets"]
                }
            }
        }
    ]
)
print(f"Security Incident Response Agent '{security_incident_response_agent.name}' created with ID: {security_incident_response_agent.id}")

# 9. Community & Governance Agent
community_governance_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for coordinating with crypto communities, participating in decentralized governance, and managing bounty programs.",
    name="community-governance-agent",
    tools=[
        {"type": "web_search"}, # For searching community forums, DAO platforms, or news related to governance.
        {
            "type": "function",
            "function": {
                "name": "track_governance_proposal",
                "description": "Track the status, voting outcomes, and discussion for a specific blockchain governance proposal.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "proposal_id": {"type": "string", "description": "Identifier of the governance proposal (e.g., 'AIP-123', 'Uniswap-Gov-4')."}
                    },
                    "required": ["proposal_id"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "submit_bounty_claim",
                "description": "Submit a claim for a bug bounty or community grant on behalf of a user or project.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "claim_id": {"type": "string", "description": "Unique identifier for the claim (if already existing)."},
                        "details": {"type": "string", "description": "Detailed description of the bounty claim (e.g., 'found critical bug in contract X')."}
                    },
                    "required": ["claim_id", "details"]
                }
            }
        }
    ]
)
print(f"Community & Governance Agent '{community_governance_agent.name}' created with ID: {community_governance_agent.id}")


# NEW AGENTS ADDED
# 10. User Support Agent
user_support_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for assisting users with crypto-related queries, troubleshooting transactions, and providing wallet support.",
    name="user-support-agent",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "get_user_transaction_history",
                "description": "Retrieves a user's transaction history for a specific cryptocurrency or blockchain.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "user_id": {"type": "string", "description": "Unique identifier for the user."},
                        "currency": {"type": "string", "description": "Optional: Specific cryptocurrency to filter transactions (e.g., 'ETH', 'USDT')."}
                    },
                    "required": ["user_id"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "troubleshoot_transaction",
                "description": "Helps diagnose issues with a pending, failed, or stuck cryptocurrency transaction.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "transaction_id": {"type": "string", "description": "The transaction hash or ID."},
                        "blockchain": {"type": "string", "description": "The blockchain network where the transaction occurred."}
                    },
                    "required": ["transaction_id", "blockchain"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "send_user_alert",
                "description": "Sends an alert or notification to a specific user regarding their crypto account or a market event.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "user_id": {"type": "string", "description": "Unique identifier for the user."},
                        "message_content": {"type": "string", "description": "The content of the alert message."},
                        "contact_method": {"type": "string", "description": "Preferred method of contact (e.g., 'email', 'in-app notification')."}
                    },
                    "required": ["user_id", "message_content", "contact_method"]
                }
            }
        }
    ]
)
print(f"User Support Agent '{user_support_agent.name}' created with ID: {user_support_agent.id}")

# 11. Liquidity Provision Agent
liquidity_provision_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for managing liquidity provision strategies, monitoring pool health, and optimizing yield in DeFi protocols.",
    name="liquidity-provision-agent",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "check_lp_health",
                "description": "Checks the health, impermanent loss status, and current APY of a specified liquidity pool.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "pool_id": {"type": "string", "description": "Identifier of the liquidity pool (e.g., 'Uniswap ETH-USDT', 'Curve USDC-DAI')."}
                    },
                    "required": ["pool_id"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "find_optimal_lp_pool",
                "description": "Searches for optimal liquidity pools based on desired asset pairs, APY range, and blockchain.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "asset_pair": {"type": "string", "description": "The asset pair for the pool (e.g., 'ETH/USDT')."},
                        "desired_apy_range": {"type": "string", "description": "Optional: Desired APY range (e.g., '5-10%', 'over 20%')."},
                        "blockchain": {"type": "string", "description": "Optional: Target blockchain (e.g., 'Ethereum', 'Polygon')."}
                    },
                    "required": ["asset_pair"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "adjust_lp_position",
                "description": "Adjusts a liquidity provision position in a specified pool (e.g., add, remove, harvest rewards).",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "pool_id": {"type": "string", "description": "Identifier of the liquidity pool."},
                        "amount": {"type": "number", "description": "The amount of liquidity or tokens to adjust."},
                        "action": {"type": "string", "description": "The action to perform ('add', 'remove', 'harvest')."}
                    },
                    "required": ["pool_id", "amount", "action"]
                }
            }
        }
    ]
)
print(f"Liquidity Provision Agent '{liquidity_provision_agent.name}' created with ID: {liquidity_provision_agent.id}")

# 12. Market Data Analyst Agent
market_data_analyst_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for real-time crypto market monitoring, price analysis, and generating trading signals.",
    name="market-data-analyst-agent",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "get_current_price",
                "description": "Retrieves the current market price of a cryptocurrency pair.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "currency_pair": {"type": "string", "description": "The currency pair (e.g., 'BTC/USD', 'ETH/BTC')."}
                    },
                    "required": ["currency_pair"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "analyze_market_sentiment",
                "description": "Analyzes market sentiment for a given cryptocurrency asset from various sources (e.g., news, social media).",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "asset_id": {"type": "string", "description": "Identifier of the cryptocurrency asset (e.g., 'BTC', 'DOGE')."},
                        "source_type": {"type": "string", "description": "Type of source for sentiment analysis (e.g., 'news', 'social media', 'all')."}
                    },
                    "required": ["asset_id", "source_type"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "generate_trading_signal",
                "description": "Generates potential buy/sell trading signals based on technical analysis or predefined strategies.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "currency_pair": {"type": "string", "description": "The currency pair for the signal (e.g., 'ETH/USD')."},
                        "analysis_type": {"type": "string", "description": "Type of analysis to perform (e.g., 'RSI', 'MACD', 'trend_following')."}
                    },
                    "required": ["currency_pair", "analysis_type"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "track_defi_metrics",
                "description": "Tracks key decentralized finance (DeFi) metrics for a given protocol (e.g., Total Value Locked, trading volume).",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "protocol_name": {"type": "string", "description": "Name of the DeFi protocol (e.g., 'Uniswap', 'Aave')."}
                    },
                    "required": ["protocol_name"]
                }
            }
        }
    ]
)
print(f"Market Data Analyst Agent '{market_data_analyst_agent.name}' created with ID: {market_data_analyst_agent.id}")

# 13. Treasury Management Agent
treasury_management_agent = client.beta.agents.create(
    model="mistral-large-latest",
    description="Agent for monitoring crypto treasury balances, tracking token flows, and optimizing fund allocation for growth or stability.",
    name="treasury-management-agent",
    tools=[
        {
            "type": "function",
            "function": {
                "name": "get_treasury_balance",
                "description": "Retrieves the balance of a specific wallet or token within the crypto treasury.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "wallet_address": {"type": "string", "description": "The wallet address associated with the treasury."},
                        "token_id": {"type": "string", "description": "Optional: Specific token to get balance for (e.g., 'ETH', 'USDT')."}
                    },
                    "required": ["wallet_address"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "log_token_transfer",
                "description": "Logs a token transfer event within the treasury for audit and tracking purposes.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "from_address": {"type": "string", "description": "Sending wallet address."},
                        "to_address": {"type": "string", "description": "Receiving wallet address."},
                        "amount": {"type": "number", "description": "Amount of tokens transferred."},
                        "token_id": {"type": "string", "description": "Identifier of the token transferred (e.g., 'USDC', 'DAI')."},
                        "tx_hash": {"type": "string", "description": "Transaction hash of the transfer."}
                    },
                    "required": ["from_address", "to_address", "amount", "token_id", "tx_hash"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "forecast_token_liquidity",
                "description": "Forecasts future liquidity needs or availability for a specific token held in the treasury.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "token_id": {"type": "string", "description": "Identifier of the token to forecast liquidity for."},
                        "time_horizon": {"type": "string", "description": "The period for the forecast (e.g., 'next quarter', 'next 6 months')."}
                    },
                    "required": ["token_id", "time_horizon"]
                }
            }
        },
        {
            "type": "function",
            "function": {
                "name": "optimize_fund_allocation",
                "description": "Recommends changes to treasury fund allocation based on a target strategy (e.g., 'diversify', 'increase yield').",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "current_portfolio_json": {"type": "string", "description": "JSON string of current treasury holdings (e.g., '[{\"token\": \"ETH\", \"amount\": 10}, {\"token\": \"USDC\", \"amount\": 5000}]')."},
                        "target_strategy": {"type": "string", "description": "The desired optimization strategy (e.g., 'diversify_risk', 'maximize_stablecoin_yield')."}
                    },
                    "required": ["current_portfolio_json", "target_strategy"]
                }
            }
        }
    ]
)
print(f"Treasury Management Agent '{treasury_management_agent.name}' created with ID: {treasury_management_agent.id}")


print("\nAll new Cryptocurrency Operations agents have been defined.")

# Mock functions for Crypto domain tools
# These mock functions provide simulated responses for the new agents' tools.

# Smart Contract Development Agent Mocks
def design_smart_contract(contract_type: str, functionalities: list, blockchain_platform: str):
    """MOCK function to design a smart contract."""
    print(f"\n[DEBUG] MOCK CALL: design_smart_contract Type: '{contract_type}', Functionalities: {functionalities}, Platform: '{blockchain_platform}'")
    if contract_type == "ERC-20" and "minting" in functionalities and blockchain_platform == "Ethereum":
        return {"status": "success", "contract_id": "SC-001", "abi_snippet": "function mint()", "message": "ERC-20 contract with minting designed for Ethereum."}
    return {"status": "success", "contract_id": "SC-GENERIC", "message": "Generic contract design drafted."}

def deploy_contract(contract_code: str, blockchain_network: str, gas_limit: int = None):
    """MOCK function to deploy a contract."""
    print(f"\n[DEBUG] MOCK CALL: deploy_contract Code: '{contract_code[:30]}...', Network: {blockchain_network}, Gas Limit: {gas_limit}")
    if blockchain_network == "mainnet" and len(contract_code) > 100:
        return {"status": "success", "transaction_hash": "0xabc123def456", "message": "Contract deployed to mainnet."}
    return {"status": "failed", "message": "Contract deployment failed (mock reason)."}

# Exchange Integration Agent Mocks
def predict_liquidity(exchange_name: str, trading_pair: str, time_window: str):
    """MOCK function to predict liquidity."""
    print(f"\n[DEBUG] MOCK CALL: predict_liquidity Exchange: {exchange_name}, Pair: {trading_pair}, Time Window: {time_window}")
    if exchange_name == "Binance" and trading_pair == "BTC/USD":
        return {"status": "success", "liquidity_level": "High", "confidence": 0.95, "message": "High liquidity predicted for BTC/USD on Binance."}
    return {"status": "moderate", "liquidity_level": "Moderate", "message": "Moderate liquidity expected."}

def check_trading_pair_availability(exchange_name: str, pair: str):
    """MOCK function to check trading pair availability."""
    print(f"\n[DEBUG] MOCK CALL: check_trading_pair_availability Exchange: {exchange_name}, Pair: {pair}")
    if exchange_name == "Coinbase" and pair.upper() == "XRP/USD":
        return {"status": "unavailable", "message": "XRP/USD is not available on Coinbase in this region."}
    return {"status": "available", "message": "Trading pair is available."}

# Risk Management Agent Mocks
def predict_risk_impact(asset_id: str, risk_type: str):
    """MOCK function to predict risk impact."""
    print(f"\n[DEBUG] MOCK CALL: predict_risk_impact Asset: {asset_id}, Risk: {risk_type}")
    if risk_type == "impermanent_loss" and asset_id == "ETH-USDT_LP":
        return {"status": "warning", "impact_level": "Medium", "potential_loss_percentage": 5.0, "message": "Potential impermanent loss predicted."}
    return {"status": "low_impact", "message": "Low risk impact expected."}

def propose_mitigation_strategy(asset_id: str, risk_details: str):
    """MOCK function to propose mitigation strategy."""
    print(f"\n[DEBUG] MOCK CALL: propose_mitigation_strategy Asset: {asset_id}, Details: '{risk_details}'")
    if "high volatility" in risk_details:
        return {"status": "strategy_proposed", "strategy": "Reduce exposure, consider hedging.", "message": "Risk mitigation strategy drafted."}
    return {"status": "strategy_proposed", "strategy": "Monitor closely.", "message": "Minor mitigation plan."}

# Portfolio Management Agent Mocks
def allocate_portfolio(total_value: float, allocation_strategy: str):
    """MOCK function to allocate portfolio."""
    print(f"\n[DEBUG] MOCK CALL: allocate_portfolio Value: {total_value}, Strategy: '{allocation_strategy}'")
    if allocation_strategy == "conservative":
        return {"status": "allocated", "breakdown": {"BTC": 0.5, "ETH": 0.3, "USDT": 0.2}, "message": "Conservative portfolio allocated."}
    return {"status": "allocated", "breakdown": {"BTC": 0.4, "ETH": 0.4, "ALT": 0.2}, "message": "Portfolio allocated."}

def simulate_investment_change(asset_id: str, amount: float, investment_period: str):
    """MOCK function to simulate investment change."""
    print(f"\n[DEBUG] MOCK CALL: simulate_investment_change Asset: {asset_id}, Amount: {amount}, Period: {investment_period}")
    if asset_id == "ETH" and amount > 0:
        return {"status": "simulation_complete", "projected_gain": 0.15 * amount, "message": "Investment simulation complete, projected gain."}
    return {"status": "simulation_complete", "projected_gain": 0.0, "message": "Investment simulation complete."}

# Blockchain Node Management Agent Mocks
def schedule_node_maintenance(node_id: str, maintenance_type: str, preferred_date: str):
    """MOCK function to schedule node maintenance."""
    print(f"\n[DEBUG] MOCK CALL: schedule_node_maintenance Node: {node_id}, Type: {maintenance_type}, Date: {preferred_date}")
    return {"status": "scheduled", "node_id": node_id, "message": "Node maintenance scheduled."}

def check_node_status(node_id: str):
    """MOCK function to check node status."""
    print(f"\n[DEBUG] MOCK CALL: check_node_status Node: {node_id}")
    if node_id == "ETH-NODE-001":
        return {"status": "synced", "health": "Operational", "last_block": 12345678, "message": "Node is synced and healthy."}
    return {"status": "syncing", "health": "Degraded", "message": "Node is syncing."}

def log_node_event(node_id: str, event_description: str, timestamp: str):
    """MOCK function to log node event."""
    print(f"\n[DEBUG] MOCK CALL: log_node_event Node: {node_id}, Event: '{event_description}', Timestamp: {timestamp}")
    return {"status": "logged", "node_id": node_id, "message": "Node event logged."}

# On-Chain Data Analysis Agent Mocks
def get_transaction_data(transaction_hash: str, blockchain: str):
    """MOCK function to retrieve transaction data."""
    print(f"\n[DEBUG] MOCK CALL: get_transaction_data Hash: {transaction_hash}, Blockchain: {blockchain}")
    if transaction_hash == "0x123abc" and blockchain == "Ethereum":
        return {"status": "success", "tx_data": {"from": "0xSender", "to": "0xReceiver", "value": 1.5, "gas_used": 21000}, "message": "Transaction data retrieved."}
    return {"status": "not_found", "message": "Transaction data not found."}

def analyze_wallet_activity(wallet_address: str, time_frame: str):
    """MOCK function to analyze wallet activity."""
    print(f"\n[DEBUG] MOCK CALL: analyze_wallet_activity Wallet: {wallet_address}, Time Frame: {time_frame}")
    # Example: Simple mock analysis
    if wallet_address == "0xTestWallet":
        return {"status": "success", "total_transactions": 15, "incoming_volume_eth": 10.2, "outgoing_volume_eth": 8.5, "message": "Wallet activity analyzed."}
    return {"status": "no_data", "message": "No significant activity found for wallet."}

# Regulatory Compliance Agent (Crypto) Mocks
def check_crypto_regulation(aspect: str, jurisdiction: str):
    """MOCK function to check crypto regulation."""
    print(f"\n[DEBUG] MOCK CALL: check_crypto_regulation Aspect: {aspect}, Jurisdiction: {jurisdiction}")
    if aspect == "AML compliance" and jurisdiction == "US FINCEN":
        return {"status": "compliant", "details": "Follows FINCEN AML guidelines.", "message": "Regulation check complete."}
    return {"status": "non_compliant", "details": "Requires further legal review.", "message": "Potential non-compliance."}

def submit_compliance_report(report_type: str, summary: str):
    """MOCK function to submit compliance report."""
    print(f"\n[DEBUG] MOCK CALL: submit_compliance_report Type: {report_type}, Summary: '{summary}'")
    return {"status": "submitted", "report_id": "CR-001", "message": "Compliance report submitted."}

# Security Incident Response Agent Mocks
def analyze_security_incident(incident_id: str, blockchain_data_json: str):
    """MOCK function to analyze security incident."""
    print(f"\n[DEBUG] MOCK CALL: analyze_security_incident Incident: {incident_id}, Data: {blockchain_data_json}")
    if incident_id == "HACK-001" and "reentrancy" in blockchain_data_json:
        return {"status": "success", "attack_vector": "Reentrancy vulnerability in contract X.", "confidence": 0.95, "message": "Incident analysis complete."}
    return {"status": "needs_more_data", "message": "Further data needed for incident analysis."}

def predict_financial_impact(incident_id: str, affected_assets: list):
    """MOCK function to predict financial impact."""
    print(f"\n[DEBUG] MOCK CALL: predict_financial_impact Incident: {incident_id}, Affected Assets: {affected_assets}")
    if incident_id == "HACK-001" and "USDT" in affected_assets:
        return {"status": "warning", "estimated_loss_usd": 500000.00, "message": "Estimated 500K USD loss for USDT holders."}
    return {"status": "no_major_impact", "message": "No significant financial impact predicted."}

# Community & Governance Agent Mocks
def track_governance_proposal(proposal_id: str):
    """MOCK function to track governance proposal."""
    print(f"\n[DEBUG] MOCK CALL: track_governance_proposal Proposal: {proposal_id}")
    if proposal_id == "AIP-123":
        return {"status": "active", "votes_for": 1200000, "votes_against": 50000, "outcome": "Passing", "message": "AIP-123 is currently passing."}
    return {"status": "not_found", "message": "Proposal not found."}

def submit_bounty_claim(claim_id: str, details: str):
    """MOCK function to submit bounty claim."""
    print(f"\n[DEBUG] MOCK CALL: submit_bounty_claim Claim ID: {claim_id}, Details: '{details}'")
    return {"status": "submitted", "claim_reference": "BC-001", "message": "Bounty claim submitted for review."}

# User Support Agent Mocks
def get_user_transaction_history(user_id: str, currency: str = None):
    """MOCK function to retrieve user transaction history."""
    print(f"\n[DEBUG] MOCK CALL: get_user_transaction_history User: {user_id}, Currency: {currency}")
    if user_id == "User-001" and currency == "ETH":
        return {"status": "success", "history": [{"tx_hash": "0x111...", "amount": 0.5, "type": "send"}, {"tx_hash": "0x222...", "amount": 1.2, "type": "receive"}], "message": "ETH transaction history retrieved."}
    return {"status": "not_found", "message": "User or currency history not found."}

def troubleshoot_transaction(transaction_id: str, blockchain: str):
    """MOCK function to troubleshoot transaction."""
    print(f"\n[DEBUG] MOCK CALL: troubleshoot_transaction Tx ID: {transaction_id}, Blockchain: {blockchain}")
    if transaction_id == "0xfailedtx" and blockchain == "Ethereum":
        return {"status": "issue_identified", "reason": "Insufficient gas limit.", "solution": "Resubmit with higher gas.", "message": "Transaction troubleshooting complete."}
    return {"status": "no_issue_found", "message": "Transaction appears normal."}

def send_user_alert(user_id: str, message_content: str, contact_method: str):
    """MOCK function to send user alert."""
    print(f"\n[DEBUG] MOCK CALL: send_user_alert User: {user_id}, Content: '{message_content}', Method: {contact_method}")
    return {"status": "sent", "user_id": user_id, "message": "Alert sent successfully."}

# Liquidity Provision Agent Mocks
def check_lp_health(pool_id: str):
    """MOCK function to check LP health."""
    print(f"\n[DEBUG] MOCK CALL: check_lp_health Pool: {pool_id}")
    if pool_id == "Uniswap ETH-USDT":
        return {"status": "healthy", "impermanent_loss_usd": 10.50, "current_apy_percent": 8.7, "message": "LP pool is healthy."}
    return {"status": "monitor", "impermanent_loss_usd": 50.00, "current_apy_percent": 3.2, "message": "LP pool needs monitoring."}

def find_optimal_lp_pool(asset_pair: str, desired_apy_range: str = None, blockchain: str = None):
    """MOCK function to find optimal LP pool."""
    print(f"\n[DEBUG] MOCK CALL: find_optimal_lp_pool Pair: {asset_pair}, APY Range: {desired_apy_range}, Blockchain: {blockchain}")
    if asset_pair == "ETH/USDT" and desired_apy_range == "5-10%":
        return {"status": "success", "optimal_pools": ["Uniswap ETH-USDT (8.7% APY)", "Sushiswap ETH-USDT (7.5% APY)"], "message": "Optimal LP pools found."}
    return {"status": "no_match", "message": "No optimal pools matching criteria."}

def adjust_lp_position(pool_id: str, amount: float, action: str):
    """MOCK function to adjust LP position."""
    print(f"\n[DEBUG] MOCK CALL: adjust_lp_position Pool: {pool_id}, Amount: {amount}, Action: {action}")
    return {"status": "executed", "pool_id": pool_id, "action": action, "message": f"LP position {action}ed successfully."}

# Market Data Analyst Agent Mocks
def get_current_price(currency_pair: str):
    """MOCK function to retrieve current crypto price."""
    print(f"\n[DEBUG] MOCK CALL: get_current_price Pair: {currency_pair}")
    if currency_pair.upper() == "BTC/USD":
        return {"status": "success", "pair": currency_pair, "price": 68500.25, "timestamp": time.time(), "message": "Current price retrieved."}
    return {"status": "not_found", "message": "Currency pair not found."}

def analyze_market_sentiment(asset_id: str, source_type: str):
    """MOCK function to analyze market sentiment."""
    print(f"\n[DEBUG] MOCK CALL: analyze_market_sentiment Asset: {asset_id}, Source: {source_type}")
    if asset_id == "BTC" and source_type == "news":
        return {"status": "success", "sentiment": "Bullish", "score": 0.75, "message": "BTC news sentiment is bullish."}
    return {"status": "neutral", "sentiment": "Neutral", "message": "Sentiment analysis inconclusive."}

def generate_trading_signal(currency_pair: str, analysis_type: str):
    """MOCK function to generate trading signal."""
    print(f"\n[DEBUG] MOCK CALL: generate_trading_signal Pair: {currency_pair}, Analysis: {analysis_type}")
    if currency_pair.upper() == "ETH/USD" and analysis_type == "RSI":
        return {"status": "signal_generated", "type": "Buy", "reason": "RSI oversold.", "message": "Buy signal generated for ETH/USD."}
    return {"status": "no_signal", "message": "No strong trading signal generated."}

def track_defi_metrics(protocol_name: str):
    """MOCK function to track DeFi metrics."""
    print(f"\n[DEBUG] MOCK CALL: track_defi_metrics Protocol: {protocol_name}")
    if protocol_name == "Uniswap":
        return {"status": "success", "tvl_usd": 5000000000.00, "daily_volume_usd": 150000000.00, "message": "Uniswap metrics retrieved."}
    return {"status": "not_found", "message": "DeFi protocol metrics not found."}

# Treasury Management Agent Mocks
def get_treasury_balance(wallet_address: str, token_id: str = None):
    """MOCK function to retrieve treasury balance."""
    print(f"\n[DEBUG] MOCK CALL: get_treasury_balance Wallet: {wallet_address}, Token: {token_id}")
    if wallet_address == "0xTreasuryMain" and token_id is None:
        return {"status": "success", "total_balance_usd": 1000000.00, "breakdown": {"ETH": 500, "USDC": 700000}, "message": "Treasury total balance retrieved."}
    elif wallet_address == "0xTreasuryMain" and token_id == "ETH":
        return {"status": "success", "token_balance": 500, "token_id": "ETH", "message": "ETH balance retrieved."}
    return {"status": "not_found", "message": "Treasury balance not found."}

def log_token_transfer(from_address: str, to_address: str, amount: float, token_id: str, tx_hash: str):
    """MOCK function to log a token transfer."""
    print(f"\n[DEBUG] MOCK CALL: log_token_transfer From: {from_address}, To: {to_address}, Amount: {amount} {token_id}, TxHash: {tx_hash}")
    return {"status": "logged", "log_id": f"TLOG-{int(time.time())}", "message": "Token transfer logged."}

def forecast_token_liquidity(token_id: str, time_horizon: str):
    """MOCK function to forecast token liquidity."""
    print(f"\n[DEBUG] MOCK CALL: forecast_token_liquidity Token: {token_id}, Time Horizon: {time_horizon}")
    if token_id == "DAO_GOV_TOKEN" and time_horizon == "next quarter":
        return {"status": "success", "forecasted_liquidity_usd": 200000.00, "message": "DAO governance token liquidity forecast generated."}
    return {"status": "success", "forecasted_liquidity_usd": 0.0, "message": "Token liquidity forecast generated."}

def optimize_fund_allocation(current_portfolio_json: str, target_strategy: str):
    """MOCK function to optimize fund allocation."""
    print(f"\n[DEBUG] MOCK CALL: optimize_fund_allocation Current Portfolio: {current_portfolio_json}, Strategy: '{target_strategy}'")
    try:
        current_portfolio = json.loads(current_portfolio_json)
        if target_strategy == "diversify_risk":
            return {"status": "recommendation", "recommended_actions": "Rebalance 10% ETH into stablecoins, add small cap altcoin exposure.", "message": "Fund allocation optimized for diversification."}
        return {"status": "recommendation", "recommended_actions": "Review strategy.", "message": "No specific optimization recommended."}
    except json.JSONDecodeError:
        return {"status": "error", "message": "Invalid JSON format for current portfolio."}


# Master Tool Executor Mapping
tool_executor = {
    "design_smart_contract": design_smart_contract,
    "deploy_contract": deploy_contract,
    "predict_liquidity": predict_liquidity,
    "check_trading_pair_availability": check_trading_pair_availability,
    "predict_risk_impact": predict_risk_impact,
    "propose_mitigation_strategy": propose_mitigation_strategy,
    "allocate_portfolio": allocate_portfolio,
    "simulate_investment_change": simulate_investment_change,
    "schedule_node_maintenance": schedule_node_maintenance,
    "check_node_status": check_node_status,
    "log_node_event": log_node_event,
    "get_transaction_data": get_transaction_data,
    "analyze_wallet_activity": analyze_wallet_activity,
    "check_crypto_regulation": check_crypto_regulation,
    "submit_compliance_report": submit_compliance_report,
    "analyze_security_incident": analyze_security_incident,
    "predict_financial_impact": predict_financial_impact,
    "track_governance_proposal": track_governance_proposal,
    "submit_bounty_claim": submit_bounty_claim,
    "get_user_transaction_history": get_user_transaction_history,
    "troubleshoot_transaction": troubleshoot_transaction,
    "send_user_alert": send_user_alert,
    "check_lp_health": check_lp_health,
    "find_optimal_lp_pool": find_optimal_lp_pool,
    "adjust_lp_position": adjust_lp_position,
    "get_current_price": get_current_price,
    "analyze_market_sentiment": analyze_market_sentiment,
    "generate_trading_signal": generate_trading_signal,
    "track_defi_metrics": track_defi_metrics,
    "get_treasury_balance": get_treasury_balance,
    "log_token_transfer": log_token_transfer,
    "forecast_token_liquidity": forecast_token_liquidity,
    "optimize_fund_allocation": optimize_fund_allocation,
    "internal_web_search_tool": lambda *args, **kwargs: "Mock web search: General crypto information retrieved."
}

# Function to standardize tools for client.chat.complete
def get_api_call_tools_list(agent_tools):
    api_tools = []
    for tool in agent_tools:
        if tool.type == 'function':
            api_tools.append(tool.model_dump())
        elif tool.type == 'web_search':
            api_tools.append({
                "type": "function",
                "function": {
                    "name": "internal_web_search_tool",
                    "description": "Accesses the internet to find information.",
                    "parameters": {
                        "type": "object",
                        "properties": {}
                    }
                }
            })
    return api_tools

# Test Case Execution for Crypto Agents
print("\n--- Executing Test Cases for Cryptocurrency Operations Agents (via chat completions) ---")

test_cases = [
    {
        "agent": smart_contract_development_agent,
        "name": "Smart Contract Development Agent",
        "query": "Design an ERC-721 NFT contract with royalty features for Polygon.",
        "expected_tool_call": "design_smart_contract"
    },
    {
        "agent": exchange_integration_agent,
        "name": "Exchange Integration Agent",
        "query": "Predict liquidity for ETH/USDT on Kraken for the next 24 hours.",
        "expected_tool_call": "predict_liquidity"
    },
    {
        "agent": risk_management_agent,
        "name": "Risk Management Agent",
        "query": "Predict risk impact for asset TOKEN-X due to potential oracle failure.",
        "expected_tool_call": "predict_risk_impact"
    },
    {
        "agent": portfolio_management_agent,
        "name": "Portfolio Management Agent",
        "query": "Allocate a portfolio of $10,000 using an aggressive strategy.",
        "expected_tool_call": "allocate_portfolio"
    },
    {
        "agent": blockchain_node_management_agent,
        "name": "Blockchain Node Management Agent",
        "query": "Schedule a software upgrade for validator node NODE-ETH-005 on 2025-07-20.",
        "expected_tool_call": "schedule_node_maintenance"
    },
    {
        "agent": on_chain_data_analysis_agent,
        "name": "On-Chain Data Analysis Agent",
        "query": "Get transaction data for hash 0x789def on the Bitcoin blockchain.",
        "expected_tool_call": "get_transaction_data"
    },
    {
        "agent": regulatory_compliance_agent_crypto,
        "name": "Regulatory Compliance Agent (Crypto)",
        "query": "Check security token offering rules for Singapore MAS.",
        "expected_tool_call": "check_crypto_regulation"
    },
    {
        "agent": security_incident_response_agent,
        "name": "Security Incident Response Agent",
        "query": "Analyze security incident EXP-123 using blockchain data: {\"exploit_type\": \"flash loan\", \"affected_contract\": \"0xDeFiLend\"}.",
        "expected_tool_call": "analyze_security_incident"
    },
    {
        "agent": community_governance_agent,
        "name": "Community & Governance Agent",
        "query": "Track governance proposal 'DAO-Vote-007'.",
        "expected_tool_call": "track_governance_proposal"
    },
    {
        "agent": user_support_agent,
        "name": "User Support Agent",
        "query": "Get transaction history for User-005 for USDC.",
        "expected_tool_call": "get_user_transaction_history"
    },
    {
        "agent": liquidity_provision_agent,
        "name": "Liquidity Provision Agent",
        "query": "Check health of Uniswap ETH-DAI pool.",
        "expected_tool_call": "check_lp_health"
    },
    {
        "agent": market_data_analyst_agent,
        "name": "Market Data Analyst Agent",
        "query": "Get current price of DOGE/USDT.",
        "expected_tool_call": "get_current_price"
    },
    {
        "agent": treasury_management_agent,
        "name": "Treasury Management Agent",
        "query": "Get balance of wallet 0xTreasuryXYZ for token WBTC.",
        "expected_tool_call": "get_treasury_balance"
    }
]

for test_case in test_cases:
    agent_to_test = test_case["agent"]
    agent_name = test_case["name"]
    user_query = test_case["query"]
    expected_tool_call_name = test_case["expected_tool_call"]

    print(f"\n--- Executing Test Case for the {agent_name} ---")
    print(f"User: {user_query}")
    conversation_history = []
    conversation_history.append({"role": "user", "content": user_query})

    try:
        api_call_tools_list = get_api_call_tools_list(agent_to_test.tools)

        print(f"[DEBUG] Sending initial user query to the {agent_name}...")
        response_turn1 = client.chat.complete(
            model=agent_to_test.model,
            messages=conversation_history,
            tools=api_call_tools_list,
        )

        assistant_message_turn1 = response_turn1.choices[0].message
        conversation_history.append(assistant_message_turn1.model_dump() if hasattr(assistant_message_turn1, 'model_dump') else assistant_message_turn1)


        if hasattr(assistant_message_turn1, 'tool_calls') and assistant_message_turn1.tool_calls:
            print(f"\n{agent_name} proposed tool calls (Turn 1):")
            for tool_call in assistant_message_turn1.tool_calls:
                print(f"  Tool Name: {tool_call.function.name}")
                print(f"  Tool Arguments (JSON string): {tool_call.function.arguments}")

                tool_output_content = None
                if tool_call.function.name in tool_executor:
                    try:
                        args = json.loads(tool_call.function.arguments)
                        tool_output = tool_executor[tool_call.function.name](**args)
                        tool_output_content = json.dumps(tool_output)
                        print(f"  [DEBUG] Local MOCK {tool_call.function.name} executed. Output: {tool_output}")
                    except json.JSONDecodeError as e:
                        print(f"  [ERROR] Failed to parse tool arguments for {tool_call.function.name}: {e}")
                        tool_output_content = json.dumps({"error": f"Failed to parse arguments: {e}"})
                    except Exception as e:
                        print(f"  [ERROR] Error executing local mock {tool_call.function.name}: {e}")
                        tool_output_content = json.dumps({"error": f"Tool execution failed: {e}"})
                else:
                    print(f"  [DEBUG] Unhandled tool call: {tool_call.function.name}")
                    tool_output_content = json.dumps({"error": "Tool not handled by client-side executor."})

                conversation_history.append(
                    {
                        "role": "tool",
                        "name": tool_call.function.name,
                        "content": tool_output_content,
                        "tool_call_id": tool_call.id
                    }
                )
                print(f"  [DEBUG] Tool output for '{tool_call.function.name}' added to history.")

            print(f"\n[DEBUG] Sending conversation history with tool outputs back for final response from {agent_name}...")
            final_response = client.chat.complete(
                model=agent_to_test.model,
                messages=conversation_history,
                tools=api_call_tools_list,
            )

            final_assistant_message = final_response.choices[0].message
            print(f"\n{agent_name}'s Final Response:")
            print(final_assistant_message.content)

            conversation_history.append(final_assistant_message.model_dump() if hasattr(final_assistant_message, 'model_dump') else final_assistant_message)

        else:
            print(f"\n{agent_name}'s initial response (no tool calls proposed):")
            print(assistant_message_turn1.content)

    except Exception as e:
        print(f"\nAn error occurred during {agent_name} interaction: {e}")
        print("Please check your API key, model availability, network connection, or SDK version.")
        print("If you continue to experience errors, a complete restart of your Python environment (e.g., Colab runtime) might help.")

print("\n--- All test cases execution complete. ---")

Mounted at /content/gdrive
Creating AI agents for Cryptocurrency Operations Domain...
Smart Contract Development Agent 'smart-contract-development-agent' created with ID: ag_0685d45181b97a218000bec85adbbcd4
Exchange Integration Agent 'exchange-integration-agent' created with ID: ag_0685d45184b07eb780003270f12eeed4
Risk Management Agent 'risk-management-agent' created with ID: ag_0685d451899572ec800044c6db65b257
Portfolio Management Agent 'portfolio-management-agent' created with ID: ag_0685d4518e087df98000703932d1ba56
Blockchain Node Management Agent 'blockchain-node-management-agent' created with ID: ag_0685d451910d786e8000a778ce5a7d1a
On-Chain Data Analysis Agent 'on-chain-data-analysis-agent' created with ID: ag_0685d45193e8736e80001534e8b0318d
Regulatory Compliance Agent (Crypto) 'regulatory-compliance-agent-crypto' created with ID: ag_0685d45196e87bef800076581d03919f
Security Incident Response Agent 'security-incident-response-agent' created with ID: ag_0685d45199bc728280003b8e77f